# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The FAIR² dataset consists of protein abundance data from mass spectrometry analysis on extracellular vesicles isolated from human mast cells. All data elements are referenced by their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset name and description
print(f"{metadata.name}\n{metadata.description}")

## 2. Data Overview
Explore the dataset to list available record sets and their fields.

All references to dataset elements (record sets, fields, columns) use their `@id`s for reliability.

In [ ]:
# List available record sets and their fields by `@id`
record_sets = dataset.record_sets
print('Record sets (@id):')
for rs in record_sets:
    print(f"- {rs['@id']} : {rs['name'] if 'name' in rs else ''}")
    # List fields in this record set
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"    Field @id: {f['@id']} - {f['name'] if 'name' in f else ''}")

# Show a preview of the json structure for the first record set, if any
if len(record_sets) > 0:
    sample_rs_id = record_sets[0]['@id']
    print(f"\nPreview of first few records in record set {sample_rs_id}:")
    it = dataset.records(record_set=sample_rs_id)
    for i, row in enumerate(it):
        pprint.pprint(row)
        if i >= 2:
            break

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All fields (columns) retain their Croissant `@id`s.

In [ ]:
# Load all record sets into Pandas DataFrames
# Use explicit @id references for record sets and fields
dataframes = {}

# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display available columns for the first record set
main_record_set_id = record_set_ids[0]
print(f"Columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Here we filter, normalize, and group data using key numeric and categorical fields (referenced by their `@id`).

Typical numeric fields for mass spectrometry datasets could include peptide counts (e.g., `cr:numberOfPeptides`), normalized abundance (`cr:normalizedAbundance`), molecular weight (`cr:MW`), and isoelectric point (`cr:pI`).

Categorical fields can be accession number (`cr:accession`) or protein description (`cr:description`).

In [ ]:
# Example: Filter on normalized abundance, normalize, then group by protein accession
# Replace field IDs as needed using actual @ids from the metadata overview
main_df = dataframes[main_record_set_id].copy()

# Example field @ids (edit as appropriate for the dataset!):
numeric_field_id = None
group_field_id = None
# Use pandas columns to infer the likely fields
print("Available columns (@id):")
print(main_df.columns.tolist())

# Heuristically choose a numeric field and group field
for col in main_df.columns:
    # Try to pick a good numeric field
    if any(s in col.lower() for s in ['abundance','peptide','mw','pi','amount','count','intensity']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_candidates = list(main_df.select_dtypes(include=['number']).columns)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
for col in main_df.columns:
    if 'accession' in col.lower() or 'protein' in col.lower():
        group_field_id = col
        break

if numeric_field_id is not None:
    # Remove null and non-numeric rows for this field
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce').notnull()].copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    threshold = filtered_df[numeric_field_id].mean()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize distributions and relationships in the dataset using Pandas and Matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in filtered_df.columns:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id} (> mean)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load and examine metadata and records using `mlcroissant` by their Croissant schema `@id`s.
- Explore available record sets and fields, extracting and previewing data.
- Conduct basic EDA: filter, normalize, group, and visualize protein/protein-group-level mass spectrometry data from extracellular vesicles.

You can now adapt the analysis for your specific scientific objectives and extend it to more comprehensive data processing or machine learning tasks with full `mlcroissant` provenance and metadata tracking.